In [25]:
test_df = pd.read_excel("../data/test.xlsx")

print("Shape teste:", test_df.shape)
print("Colunas:", test_df.columns.tolist())

test_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/test.xlsx'

In [24]:
FINAL_MODEL_PATH = "../models/bertimbau_final_best"

final_trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)

print("Modelo final salvo em:", FINAL_MODEL_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo final salvo em: ../models/bertimbau_final_best


In [23]:
final_trainer.train()

Step,Training Loss
100,4.403756
200,4.332139
300,4.312978
400,4.257539
500,4.238328
600,4.234575
700,4.241933
800,4.194427
900,4.219865
1000,4.193143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3768, training_loss=3.909996449820808, metrics={'train_runtime': 911.2417, 'train_samples_per_second': 66.147, 'train_steps_per_second': 4.135, 'total_flos': 7588676082639456.0, 'train_loss': 3.909996449820808, 'epoch': 3.0})

In [22]:
final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_tokenized,
    data_collator=data_collator
)

In [21]:
final_training_args = TrainingArguments(
    output_dir="../models/bertimbau_final",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    num_train_epochs=3,

    weight_decay=0.01,

    save_strategy="epoch",
    save_total_limit=1,

    fp16=True,

    logging_steps=100,

    report_to="none",

    seed=SEED
)

In [20]:
eval_strategy="epoch"
load_best_model_at_end=True

In [19]:
final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

final_model = final_model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

In [18]:
full_tokenized = full_dataset.map(
    tokenize_batch,
    batched=True
)

print(full_tokenized)

Map:   0%|          | 0/20092 [00:00<?, ? examples/s]

Dataset({
    features: ['resp_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 20092
})


In [17]:
full_df = df.copy()

full_df["label"] = full_df["clarity"].map(label2id)

full_dataset = Dataset.from_pandas(
    full_df[
        ["resp_text", "label"]
    ].reset_index(drop=True)
)

print(full_dataset)

Dataset({
    features: ['resp_text', 'label'],
    num_rows: 20092
})


In [16]:
DEV_MODEL_PATH = "../models/bertimbau_dev_best"

trainer.save_model(
    DEV_MODEL_PATH
)

tokenizer.save_pretrained(
    DEV_MODEL_PATH
)

print("Modelo salvo em:", DEV_MODEL_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: ../models/bertimbau_dev_best


In [15]:
pred_output = trainer.predict(
    val_tokenized
)

pred_ids = pred_output.predictions.argmax(axis=-1)

print(
    classification_report(
        val_df["label"],
        pred_ids,
        target_names=["c1", "c234", "c5"]
    )
)

print("Matriz de confusão:")
print(
    confusion_matrix(
        val_df["label"],
        pred_ids
    )
)

              precision    recall  f1-score   support

          c1       0.49      0.55      0.52      1269
        c234       0.43      0.34      0.38      1371
          c5       0.51      0.55      0.53      1379

    accuracy                           0.48      4019
   macro avg       0.47      0.48      0.48      4019
weighted avg       0.47      0.48      0.47      4019

Matriz de confusão:
[[697 297 275]
 [443 472 456]
 [291 332 756]]


In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,4.226848,1.031912,0.464792,0.448661
2,3.975561,1.024348,0.472506,0.470788
3,3.546871,1.048215,0.478975,0.475246


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3015, training_loss=3.9684614039179102, metrics={'train_runtime': 775.5601, 'train_samples_per_second': 62.173, 'train_steps_per_second': 3.888, 'total_flos': 6077309444164728.0, 'train_loss': 3.9684614039179102, 'epoch': 3.0})

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [12]:
training_args = TrainingArguments(
    output_dir="../models/bertimbau_dev",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=4,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    fp16=True,

    logging_steps=100,
    save_total_limit=1,

    report_to="none",

    seed=SEED
)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

model = model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

In [10]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = logits.argmax(axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(
            labels,
            preds,
            average="macro"
        )
    }

In [9]:
MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["resp_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_dataset.map(
    tokenize_batch,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_batch,
    batched=True
)

Map:   0%|          | 0/16073 [00:00<?, ? examples/s]

Map:   0%|          | 0/4019 [00:00<?, ? examples/s]

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

train_dataset = Dataset.from_pandas(
    train_df[
        ["resp_text", "label"]
    ].reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df[
        ["resp_text", "label"]
    ].reset_index(drop=True)
)

In [7]:
label2id = {
    "c1": 0,
    "c234": 1,
    "c5": 2
}

id2label = {
    0: "c1",
    1: "c234",
    2: "c5"
}

train_df["label"] = train_df["clarity"].map(label2id)
val_df["label"] = val_df["clarity"].map(label2id)

In [6]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

train_idx, val_idx = next(
    sgkf.split(
        X=df,
        y=df["clarity"],
        groups=df["text_group"]
    )
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)

Treino: (16073, 5)
Validação: (4019, 5)


In [5]:
df["text_group"] = pd.factorize(
    df["resp_text"]
)[0]

In [4]:
print("Duplicatas adicionais:", df["resp_text"].duplicated().sum())
print("Textos únicos:", df["resp_text"].nunique())

Duplicatas adicionais: 1660
Textos únicos: 18432


In [3]:
df["n_chars"] = df["resp_text"].str.len()
df["n_words"] = df["resp_text"].str.split().str.len()

print(
    df.groupby("clarity")[["n_chars", "n_words"]]
    .agg(["mean", "median"])
)

             n_chars            n_words       
                mean median        mean median
clarity                                       
c1       1055.373090  782.0  152.424610  111.0
c234     1089.539180  794.0  157.059974  112.0
c5        861.085752  606.5  124.055136   87.0


In [2]:
df = pd.read_excel("../data/train.xlsx")

df["resp_text"] = df["resp_text"].astype(str)

print("Shape:", df.shape)
print(df["clarity"].value_counts())
print()
df.head()

Shape: (20092, 2)
clarity
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64



,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


In [1]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

SEED = 42

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda
